In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from numba import njit

# ==========================================
# 1. Initialization
# ==========================================

@njit
def initialize_lattice(L):
    lattice = np.empty((L, L), dtype=np.int8)

    for i in range(L):
        for j in range(L):
            lattice[i, j] = 1 if np.random.random() < 0.5 else -1

    return lattice


# ==========================================
# 2. Optimized Metropolis Sweep
# ==========================================

@njit
def _metropolis_sweep_fast(lattice, T, h, J, spin_sum):
    L = lattice.shape[0]

    for _ in range(L * L):
        x = np.random.randint(0, L)
        y = np.random.randint(0, L)

        s = lattice[x, y]

        neighbors = (
            lattice[(x + 1) % L, y]
            + lattice[(x - 1) % L, y]
            + lattice[x, (y + 1) % L]
            + lattice[x, (y - 1) % L]
        )

        dE = 2.0 * s * (J * neighbors + h)

        if dE <= 0.0 or np.random.random() < np.exp(-dE / T):
            lattice[x, y] = -s
            spin_sum -= 2 * s

    return spin_sum


# Optional: compatible standalone sweep function
@njit
def metropolis_sweep(lattice, T, h=0.0, J=1.0):
    _metropolis_sweep_fast(lattice, T, h, J, 0)
    return lattice


# ==========================================
# 3. External Magnetic Field
# ==========================================

@njit
def External_Field(t, h0, period):
    return h0 * np.sin(2.0 * np.pi * t / period)


# ==========================================
# 4. Compiled Simulation Engine
# ==========================================

@njit
def _simulation_core(lattice, random_start,
                     T, h0, period, n, J, seed):

    # Seed Numba's internal random generator
    np.random.seed(seed)

    L = lattice.shape[0]
    N = L * L

    if random_start:
        lattice = initialize_lattice(L)

    spin_sum = 0
    for i in range(L):
        for j in range(L):
            spin_sum += lattice[i, j]

    total_mcs = n * period

    h_values = np.empty(total_mcs, dtype=np.float64)
    m_values = np.empty(total_mcs, dtype=np.float64)

    for t in range(total_mcs):
        h = External_Field(t, h0, period)

        spin_sum = _metropolis_sweep_fast(
            lattice, T, h, J, spin_sum
        )

        h_values[t] = h
        m_values[t] = spin_sum / N

    return h_values, m_values


# ==========================================
# 5. User-Facing Simulator
# ==========================================

def simulator(L=40, T=2.0, h0=0.5, period=100,
              n=10, lattice=None, seed=None, J=1.0):

    if L < 2 or T <= 0 or period < 1 or n < 1:
        raise ValueError("Check L, T, period and n.")

    if seed is None:
        seed = int(
            np.random.default_rng().integers(0, 2**31)
        )

    if lattice is None:
        initial = np.empty((L, L), dtype=np.int8)
        random_start = True
    else:
        initial = np.asarray(lattice)

        if initial.shape != (L, L):
            raise ValueError("Initial lattice shape must be (L, L).")

        if not np.all((initial == 1) | (initial == -1)):
            raise ValueError("Spins must be +1 or -1.")

        initial = np.array(initial, dtype=np.int8, copy=True)
        random_start = False

    return _simulation_core(
        initial, random_start,
        T, h0, period, n, J, int(seed)
    )


In [ ]:
# ==========================================
# Demo: Temperature & External Field Response
# Animated Lattices + h(t) + m(t)
# ==========================================

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display

# ==========================================
# 1. Parameters
# ==========================================

L_demo = 40

temperatures_demo = [1.5, 2.0, 3.0]

h_initial_demo = 0.0
h_final_demo = -0.5

switch_time_demo = 45
n_sweeps_demo = 180
snapshot_interval_demo = 3

# ==========================================
# 2. Initialize Lattices
# ==========================================

lattices_demo = [
    np.ones((L_demo, L_demo), dtype=np.int8)
    for T in temperatures_demo
]

snapshots_demo = [
    [lattice.copy()] for lattice in lattices_demo
]

magnetizations_demo = [
    [np.mean(lattice)] for lattice in lattices_demo
]

times_demo = [0]
fields_demo = [h_initial_demo]

# ==========================================
# 3. Run Simulations
# ==========================================

for t in range(1, n_sweeps_demo + 1):

    # Apply negative field after switch time
    h = (
        h_initial_demo
        if t < switch_time_demo
        else h_final_demo
    )

    for i, T in enumerate(temperatures_demo):

        lattices_demo[i] = metropolis_sweep(
            lattices_demo[i],
            T,
            h=h
        )

    if t % snapshot_interval_demo == 0:

        times_demo.append(t)
        fields_demo.append(h)

        for i in range(len(temperatures_demo)):
            snapshots_demo[i].append(
                lattices_demo[i].copy()
            )

            magnetizations_demo[i].append(
                np.mean(lattices_demo[i])
            )

# ==========================================
# 4. Create Figure
# ==========================================

fig = plt.figure(figsize=(14, 8))

gs = fig.add_gridspec(
    2, 3,
    height_ratios=[1, 1],
    hspace=0.4,
    wspace=0.3
)

# Three lattice panels
lattice_axes = []
images = []

for i, T in enumerate(temperatures_demo):

    ax = fig.add_subplot(gs[0, i])

    im = ax.imshow(
        snapshots_demo[i][0],
        cmap="coolwarm",
        vmin=-1,
        vmax=1,
        interpolation="nearest"
    )

    ax.set_title(f"T = {T}")
    ax.set_xticks([])
    ax.set_yticks([])

    lattice_axes.append(ax)
    images.append(im)

# External field panel
ax_h = fig.add_subplot(gs[1, 0])

ax_h.set_xlim(0, n_sweeps_demo)
ax_h.set_ylim(
    min(h_initial_demo, h_final_demo) - 0.2,
    max(h_initial_demo, h_final_demo) + 0.2
)

ax_h.set_xlabel("MCS")
ax_h.set_ylabel("External Field h")
ax_h.set_title("External Field")
ax_h.grid(alpha=0.3)

h_line, = ax_h.step(
    [], [],
    where="post",
    color="black",
    linewidth=2
)

ax_h.axvline(
    switch_time_demo,
    linestyle="--",
    color="gray",
    alpha=0.5
)

# Magnetization panel
ax_m = fig.add_subplot(gs[1, 1:])

ax_m.set_xlim(0, n_sweeps_demo)
ax_m.set_ylim(-1.05, 1.05)

ax_m.set_xlabel("Monte Carlo Sweeps (MCS)")
ax_m.set_ylabel("Magnetization m")
ax_m.set_title("Magnetization Response")
ax_m.grid(alpha=0.3)

ax_m.axvline(
    switch_time_demo,
    linestyle="--",
    color="gray",
    alpha=0.5
)

m_lines = []

for T in temperatures_demo:
    line, = ax_m.plot(
        [], [],
        linewidth=2,
        label=f"T = {T}"
    )
    m_lines.append(line)

ax_m.legend()

title = fig.suptitle(
    "Temperature and External Field Response",
    fontsize=15
)

# ==========================================
# 5. Animation Update
# ==========================================

def update_demo(frame):

    current_time = times_demo[frame]
    current_h = fields_demo[frame]

    # Update the three lattices
    for i, T in enumerate(temperatures_demo):

        images[i].set_data(
            snapshots_demo[i][frame]
        )

        current_m = magnetizations_demo[i][frame]

        lattice_axes[i].set_title(
            f"T = {T}\n"
            f"m = {current_m:.3f}"
        )

        # Update corresponding m(t)
        m_lines[i].set_data(
            times_demo[:frame + 1],
            magnetizations_demo[i][:frame + 1]
        )

    # Update external field
    h_line.set_data(
        times_demo[:frame + 1],
        fields_demo[:frame + 1]
    )

    title.set_text(
        f"MCS = {current_time} | "
        f"External Field h = {current_h:.2f}"
    )

    return images + m_lines + [h_line, title]


# ==========================================
# 6. Display Animation
# ==========================================

ani_demo = FuncAnimation(
    fig,
    update_demo,
    frames=len(times_demo),
    interval=120,
    blit=False,
    repeat=True
)

plt.close(fig)

display(HTML(ani_demo.to_jshtml()))

In [ ]:
# ==========================================================
# Multi-Cycle Hysteresis Animation
# Simple Version + Selectable Driving Period
# ==========================================================

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets

from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display, clear_output


# ==========================================================
# 1. Parameters
# ==========================================================

L_multi_demo = 40
T_multi_demo = 2.0
h0_multi_demo = 0.5

periods_multi_demo = [10, 50, 125, 300]

n_warmup_cycles = 15
n_show_cycles = 3

seed_multi_demo = 101

# Maximum animation frames per cycle
frames_per_cycle = 20


# ==========================================================
# 2. Generate Data for Selected Period
# ==========================================================

def generate_multi_cycle_data(P):

    n_total_cycles = n_warmup_cycles + n_show_cycles

    h_all, m_all = simulator(
        L=L_multi_demo,
        T=T_multi_demo,
        h0=h0_multi_demo,
        period=P,
        n=n_total_cycles,
        seed=seed_multi_demo
    )

    n_points = n_show_cycles * P

    # Select last three complete cycles
    h_show = np.asarray(h_all[-n_points:])
    m_show = np.asarray(m_all[-n_points:])

    t_show = np.arange(n_points)

    return t_show, h_show, m_show


# ==========================================================
# 3. Build Animation
# ==========================================================

def build_multi_cycle_animation(P):

    t, h, m = generate_multi_cycle_data(P)

    n_points = len(t)

    # ------------------------------------------------------
    # Figure
    # ------------------------------------------------------

    fig, (ax_time, ax_loop) = plt.subplots(
        1, 2,
        figsize=(12, 5)
    )

    # ------------------------------------------------------
    # Left: h(t) and m(t)
    # ------------------------------------------------------

    h_line, = ax_time.plot(
        [], [],
        '--',
        color='darkorange',
        linewidth=2,
        label='External Field h'
    )

    m_line, = ax_time.plot(
        [], [],
        color='steelblue',
        linewidth=2,
        label='Magnetization m'
    )

    time_marker = ax_time.axvline(
        0,
        color='black',
        alpha=0.45,
        linestyle='--'
    )

    ax_time.set_xlim(0, n_points)
    ax_time.set_ylim(-1.1, 1.1)

    ax_time.set_xlabel('MCS')
    ax_time.set_ylabel('h(t), m(t)')
    ax_time.set_title('External Field and Magnetization')

    ax_time.grid(alpha=0.25)
    ax_time.legend(loc='lower left')

    # Mark the boundaries between cycles
    for c in range(1, n_show_cycles):

        ax_time.axvline(
            c * P,
            color='gray',
            linestyle=':',
            alpha=0.5
        )

    # ------------------------------------------------------
    # Right: Hysteresis Loops
    # ------------------------------------------------------

    colors = plt.cm.tab10(
        np.arange(n_show_cycles)
    )

    loop_lines = []

    for c in range(n_show_cycles):

        line, = ax_loop.plot(
            [], [],
            color=colors[c],
            linewidth=2,
            label=f'Cycle {c + 1}'
        )

        loop_lines.append(line)

    loop_point, = ax_loop.plot(
        [], [],
        'ko',
        markersize=6
    )

    ax_loop.set_xlim(
        -1.2 * h0_multi_demo,
        1.2 * h0_multi_demo
    )

    ax_loop.set_ylim(-1.05, 1.05)

    ax_loop.set_xlabel('External Field h')
    ax_loop.set_ylabel('Magnetization m')
    ax_loop.set_title('Hysteresis Loops')

    ax_loop.grid(alpha=0.25)
    ax_loop.legend(loc='lower right')

    title = fig.suptitle(
        f'T = {T_multi_demo} | P = {P} | L = {L_multi_demo}'
    )

    plt.tight_layout()

    # ======================================================
    # 4. Animation Update
    # ======================================================

    def update(frame):

        end = frame + 1

        # ---- Time series ----

        h_line.set_data(
            t[:end],
            h[:end]
        )

        m_line.set_data(
            t[:end],
            m[:end]
        )

        time_marker.set_xdata(
            [frame, frame]
        )

        # ---- Hysteresis loops ----

        for c in range(n_show_cycles):

            start = c * P
            stop = min(end, (c + 1) * P)

            if stop > start:

                loop_lines[c].set_data(
                    h[start:stop],
                    m[start:stop]
                )

            else:

                loop_lines[c].set_data([], [])

        # ---- Current point ----

        loop_point.set_data(
            [h[frame]],
            [m[frame]]
        )

        current_cycle = frame // P + 1

        title.set_text(
            f'T = {T_multi_demo} | '
            f'P = {P} | '
            f'Cycle = {current_cycle}/{n_show_cycles} | '
            f'MCS = {frame}'
        )

        return (
            h_line,
            m_line,
            time_marker,
            loop_point,
            *loop_lines
        )

    # ======================================================
    # 5. Select Animation Frames
    # ======================================================

    frames = []

    for c in range(n_show_cycles):

        start = c * P
        end = (c + 1) * P - 1

        cycle_frames = np.linspace(
            start,
            end,
            num=min(P, frames_per_cycle),
            dtype=int
        )

        frames.extend(cycle_frames)

    frames = np.unique(frames)

    # ======================================================
    # 6. Create Animation
    # ======================================================

    ani = FuncAnimation(
        fig,
        update,
        frames=frames,
        interval=120,
        blit=False,
        repeat=True
    )

    plt.close(fig)

    return HTML(
        ani.to_jshtml(
            fps=8,
            default_mode='loop'
        )
    )


# ==========================================================
# 7. Period Selector
# ==========================================================

period_selector_multi = widgets.Dropdown(
    options=periods_multi_demo,
    value=125,
    description='Period P:',
    layout=widgets.Layout(width='220px')
)

animation_output_multi = widgets.Output()


def update_selected_period(change=None):

    P = int(period_selector_multi.value)

    with animation_output_multi:

        clear_output(wait=True)

        display(
            build_multi_cycle_animation(P)
        )


period_selector_multi.observe(
    update_selected_period,
    names='value'
)

# ==========================================================
# 8. Display
# ==========================================================

display(
    widgets.VBox([
        period_selector_multi,
        animation_output_multi
    ])
)

update_selected_period()

In [ ]:
def loop_area(h, m):
    h = np.asarray(h)
    m = np.asarray(m)

    dh = np.roll(h, -1) - h
    m_avg = (m + np.roll(m, -1)) / 2

    return abs(np.sum(m_avg * dh))
periods = [10, 20, 50, 75, 100, 125, 150, 200, 300]
seeds = [101, 202, 303, 404, 505]

L = 40
T = 2.0
h0 = 0.5

n_cycles = 30
burn_in = 10

# Store results: rows = seeds, columns = periods
all_areas = np.zeros((len(seeds), len(periods)))

# Run simulations
for i, seed in enumerate(seeds):
    for j, P in enumerate(periods):

        h, m = simulator(
            L=L, T=T, h0=h0,
            period=P, n=n_cycles, seed=seed
        )

        cycle_areas = []

        for k in range(burn_in, n_cycles):
            start = k * P
            end = (k + 1) * P

            A = loop_area(h[start:end], m[start:end])
            cycle_areas.append(A)

        all_areas[i, j] = np.mean(cycle_areas)

# Ensemble statistics
mean_areas = np.mean(all_areas, axis=0)

area_errors = (
    np.std(all_areas, axis=0, ddof=1)
    / np.sqrt(len(seeds))
)

# Plot
plt.figure(figsize=(9, 6))

# Individual seed results
for i, seed in enumerate(seeds):
    plt.scatter(
        periods, all_areas[i],
        alpha=0.45, s=30,
        label=f"Seed {seed}"
    )

# Ensemble mean + error bars
plt.errorbar(
    periods, mean_areas,
    yerr=area_errors,
    fmt='ko-', capsize=4,
    linewidth=1.5,
    label="Mean ± SEM"
)

plt.xlabel("Driving Period P (MCS)")
plt.ylabel("Hysteresis Loop Area A")
plt.title("Hysteresis Area vs Driving Period")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
periods_s = [10, 125, 300] #some certain periods that we are interested in

plt.figure(figsize=(8, 6))

for P in periods_s:
    h, m = simulator(
        L=40, T=2.0, h0=0.5,
        period=P, n=30, seed=101
    )

    plt.plot(h[-P:], m[-P:], label=f"P = {P}")

plt.xlabel("External Field h")
plt.ylabel("Magnetization m")
plt.legend()
plt.grid()
plt.show()

In [ ]:
# ==================================================
# PERIOD SWEEP EXPLORER — High Quality Version
# ==================================================

import numpy as np
import time

L_sweep_demo = 40
T_sweep_demo = 2.0
h0_sweep_demo = 0.5

# Denser sampling of P
periods_sweep_demo = list(np.unique(np.r_[
    np.arange(10, 60, 5),
    np.arange(60, 201, 10),
    np.arange(220, 321, 20)
]))

seeds_sweep_demo = [101, 202, 303]

n_cycles_sweep_demo = 24
burn_in_sweep_demo = 12

loop_data_sweep = {}
mean_areas_sweep = []

start_time = time.perf_counter()

for P in periods_sweep_demo:

    # Collect phase-aligned cycles from all seeds
    all_cycles = []

    for seed in seeds_sweep_demo:

        h, m = simulator(
            L=L_sweep_demo,
            T=T_sweep_demo,
            h0=h0_sweep_demo,
            period=int(P),
            n=n_cycles_sweep_demo,
            seed=seed
        )

        h = np.asarray(h)
        m = np.asarray(m)

        # Shape: (number of cycles, P)
        m_cycles = m.reshape(n_cycles_sweep_demo, P)

        # Remove transient cycles
        steady_cycles = m_cycles[burn_in_sweep_demo:]

        all_cycles.append(steady_cycles)

    # Combine all seeds and cycles
    all_cycles = np.concatenate(all_cycles, axis=0)

    # Average magnetization at each driving phase
    m_mean = np.mean(all_cycles, axis=0)

    # The field is identical for every cycle
    h_cycle = h[-P:].copy()

    # Area of the phase-averaged loop
    dh = np.roll(h_cycle, -1) - h_cycle
    m_mid = (m_mean + np.roll(m_mean, -1)) / 2

    W_mean = -np.sum(m_mid * dh)
    A_mean = abs(W_mean)

    loop_data_sweep[P] = {
        "h": h_cycle,
        "m": m_mean,
        "raw_m": all_cycles[-1].copy(),
        "A_mean": A_mean,
        "W_mean": W_mean
    }

    mean_areas_sweep.append(A_mean)

    print(
        f"P = {P:3d} | "
        f"A = {A_mean:.4f}",
        flush=True
    )

mean_areas_sweep = np.asarray(mean_areas_sweep)

print(
    f"\nCompleted in "
    f"{time.perf_counter() - start_time:.2f} seconds"
)

In [ ]:
# ==================================================
# PERIOD SWEEP EXPLORER — Part 2
# Interactive Loop Morphology + A(P)
# ==================================================

import ipywidgets as widgets
from IPython.display import display
from io import BytesIO

# ==================================================
# 1. Render one image per Period
# ==================================================

rendered_sweep_images = {}

for selected_P in periods_sweep_demo:

    data = loop_data_sweep[selected_P]

    fig, axes = plt.subplots(
        1, 2,
        figsize=(10, 4.5)
    )

    ax_loop, ax_area = axes

    # ----------------------------------------------
    # Left: Hysteresis Loop
    # ----------------------------------------------

    ax_loop.plot(
        data["h"],
        data["m"],
        color="teal",
        linewidth=2
    )

    # Mark starting and ending points
    ax_loop.scatter(
        data["h"][0],
        data["m"][0],
        color="green",
        marker="o",
        s=50,
        label="Start"
    )

    ax_loop.scatter(
        data["h"][-1],
        data["m"][-1],
        color="red",
        marker="x",
        s=50,
        label="End"
    )

    ax_loop.set_xlim(
        -1.2 * h0_sweep_demo,
        1.2 * h0_sweep_demo
    )

    ax_loop.set_ylim(-1.05, 1.05)

    ax_loop.set_xlabel("External Field h")
    ax_loop.set_ylabel("Magnetization m")
    ax_loop.set_title(f"Hysteresis Loop | P = {selected_P}")
    ax_loop.grid(alpha=0.3)
    ax_loop.legend(loc="lower right")

    # ----------------------------------------------
    # Right: Area vs Period
    # ----------------------------------------------

    ax_area.plot(
        periods_sweep_demo,
        mean_areas_sweep,
        "o-",
        color="darkorange",
        linewidth=2,
        markersize=5
    )

    # Highlight selected period
    selected_area = data["A_mean"]

    ax_area.scatter(
        [selected_P],
        [selected_area],
        color="red",
        s=110,
        zorder=5
    )

    ax_area.axvline(
        selected_P,
        color="red",
        linestyle="--",
        alpha=0.4
    )

    ax_area.set_xlim(0, max(periods_sweep_demo) + 20)
    ax_area.set_ylim(
        0,
        max(2.0, np.max(mean_areas_sweep) * 1.15)
    )

    ax_area.set_xlabel("Driving Period P (MCS)")
    ax_area.set_ylabel("Mean Hysteresis Area")
    ax_area.set_title("Area vs Driving Period")
    ax_area.grid(alpha=0.3)

    fig.suptitle(
        f"T = {T_sweep_demo} | "
        f"L = {L_sweep_demo} | "
        f"P = {selected_P} | "
        f"Mean A = {selected_area:.3f}",
        fontsize=13
    )

    fig.tight_layout()

    # Save figure into memory
    buffer = BytesIO()

    fig.savefig(
        buffer,
        format="png",
        dpi=110
    )

    rendered_sweep_images[selected_P] = buffer.getvalue()

    buffer.close()
    plt.close(fig)

# ==================================================
# 2. Create Interactive Controls
# ==================================================

period_slider_sweep = widgets.SelectionSlider(
    options=[
        (f"P = {P}", P)
        for P in periods_sweep_demo
    ],
    value=10,
    description="Period:",
    continuous_update=False,
    layout=widgets.Layout(width="650px")
)

# Pre-rendered image
sweep_image = widgets.Image(
    value=rendered_sweep_images[10],
    format="png",
    layout=widgets.Layout(width="100%")
)

# ==================================================
# 3. Update Display — No Recalculation
# ==================================================

def update_period_sweep(change):

    P = change["new"]

    # Only replace image bytes
    sweep_image.value = rendered_sweep_images[P]


period_slider_sweep.observe(
    update_period_sweep,
    names="value"
)

# ==================================================
# 4. Optional Auto-Play Control
# ==================================================

play_sweep = widgets.Play(
    value=0,
    min=0,
    max=len(periods_sweep_demo) - 1,
    step=1,
    interval=1200
)

# Play controls the selected Period index
widgets.jslink(
    (play_sweep, "value"),
    (period_slider_sweep, "index")
)

# ==================================================
# 5. Display
# ==================================================

display(
    widgets.VBox([
        widgets.HBox([
            play_sweep,
            period_slider_sweep
        ]),
        sweep_image
    ])
)

In [ ]:
def calculate_Q(m, period):
    m = np.asarray(m)
    return m.reshape(-1, period).mean(axis=1)
# Store Q results for different seeds and periods
all_Q = np.zeros((len(seeds), len(periods)))

for i, seed in enumerate(seeds):
    for j, P in enumerate(periods):

        h, m = simulator(
            L=L, T=T, h0=h0,
            period=P, n=n_cycles, seed=seed
        )

        Q_values = calculate_Q(m, P)

        # Discard transient cycles
        Q_steady = Q_values[burn_in:]

        all_Q[i, j] = np.mean(np.abs(Q_steady))

# Ensemble statistics
mean_Q = np.mean(all_Q, axis=0)

Q_errors = (
    np.std(all_Q, axis=0, ddof=1)
    / np.sqrt(len(seeds))
)

# Plot
plt.figure(figsize=(8, 5))

for i, seed in enumerate(seeds):
    plt.scatter(periods, all_Q[i], alpha=0.35)

plt.errorbar(
    periods, mean_Q,
    yerr=Q_errors,
    fmt='ko', capsize=4,
    label='Mean |Q| ± SEM'
)

plt.xlabel("Driving Period P (MCS)")
plt.ylabel("Mean |Q|")
plt.title("Dynamic Order Parameter vs Driving Period")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
temperatures = [1.2, 1.5, 1.8, 2.0, 2.2, 2.3, 2.5, 2.8, 3.2]
seeds = [101, 202, 303, 404, 505]

# Shape: (periods, temperatures, seeds)
shape = (len(periods_s), len(temperatures), len(seeds))

all_areas = np.zeros(shape)
all_Q = np.zeros(shape)

for i, P in enumerate(periods_s):
    for j, T in enumerate(temperatures):
        for s, seed in enumerate(seeds):

            h, m = simulator(
                L=L, T=T, h0=h0,
                period=P, n=n_cycles,
                seed=seed
            )

            # ----- Hysteresis Area -----
            cycle_areas = []

            for k in range(burn_in, n_cycles):
                start = k * P
                end = (k + 1) * P

                A = loop_area(h[start:end], m[start:end])
                cycle_areas.append(A)

            all_areas[i, j, s] = np.mean(cycle_areas)

            # ----- Dynamic Order Parameter -----
            m_steady = np.asarray(m[burn_in * P:])
            m_cycles = m_steady.reshape(-1, P)

            Q_cycles = np.mean(m_cycles, axis=1)
            all_Q[i, j, s] = np.mean(np.abs(Q_cycles))


# ==========================================
# Ensemble Statistics
# ==========================================

mean_A = np.mean(all_areas, axis=2)
error_A = np.std(all_areas, axis=2, ddof=1) / np.sqrt(len(seeds))

mean_Q = np.mean(all_Q, axis=2)
error_Q = np.std(all_Q, axis=2, ddof=1) / np.sqrt(len(seeds))


# ==========================================
# Plot: A(T) and |Q|(T)
# ==========================================

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for i, P in enumerate(periods_s):

    axes[0].errorbar(
        temperatures, mean_A[i],
        yerr=error_A[i],
        fmt='o-', capsize=3,
        label=f'P = {P}'
    )

    axes[1].errorbar(
        temperatures, mean_Q[i],
        yerr=error_Q[i],
        fmt='o-', capsize=3,
        label=f'P = {P}'
    )

for ax in axes:
    ax.axvline(2.269, color='gray',
               linestyle='--', alpha=0.5,
               label='Zero-field Tc')
    ax.set_xlabel("Temperature T")
    ax.grid(alpha=0.3)
    ax.legend()

axes[0].set_ylabel("Hysteresis Loop Area A")
axes[0].set_title("Area vs Temperature")

axes[1].set_ylabel("Mean |Q|")
axes[1].set_title("Dynamic Order Parameter vs Temperature")

plt.tight_layout()
plt.show()

In [ ]:
temperatures_s = [1.5, 1.8, 2.0, 2.3]
periods_t = [10, 20, 50, 75, 100, 125, 150, 200, 300, 450]
seeds = [101, 202, 303, 404, 505]

L = 40
h0 = 0.5
n_cycles = 30
burn_in = 10

# Shape: (Temperature, Period, Seed)
all_areas = np.zeros((
    len(temperatures_s),
    len(periods_t),
    len(seeds)
))

for i, T in enumerate(temperatures_s):
    for j, P in enumerate(periods_t):
        for k, seed in enumerate(seeds):

            h, m = simulator(
                L=L, T=T, h0=h0,
                period=P, n=n_cycles,
                seed=seed
            )

            cycle_areas = []

            for cycle in range(burn_in, n_cycles):
                start = cycle * P
                end = (cycle + 1) * P

                A = loop_area(
                    h[start:end],
                    m[start:end]
                )

                cycle_areas.append(A)

            all_areas[i, j, k] = np.mean(cycle_areas)

mean_A = np.mean(all_areas, axis=2)
error_A = np.std(all_areas, axis=2, ddof=1) / np.sqrt(len(seeds))

plt.figure(figsize=(9, 6))

for i, T in enumerate(temperatures_s):
    plt.errorbar(
        periods_t,
        mean_A[i],
        yerr=error_A[i],
        fmt='o-',
        capsize=3,
        label=f'T = {T}'
    )

plt.xlabel("Driving Period P (MCS)")
plt.ylabel("Hysteresis Loop Area A")
plt.title("Hysteresis Area vs Driving Period at Different Temperatures")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# ==========================================================
# Multi-Temperature Period Explorer — Part 1
# Precompute Phase-Averaged Hysteresis Loops
# ==========================================================

import numpy as np
import matplotlib.pyplot as plt
import time

# ----------------------------------------------------------
# 1. Experimental Parameters
# ----------------------------------------------------------

L_TP_demo = 40
h0_TP_demo = 0.5

temperatures_TP_demo = [1.5, 1.8, 2.0, 2.3]

periods_TP_demo = [
    10, 20, 30, 40, 50, 60, 70, 80,
    90, 100, 110, 120, 125, 130, 140, 150,
    175, 200, 225, 250, 275, 300
]

n_cycles_TP_demo = 20
burn_in_TP_demo = 10
seed_TP_demo = 101

# Data storage: data[P][T]
data_TP_demo = {}

start_time = time.perf_counter()

# ----------------------------------------------------------
# 2. Precompute Trajectories
# ----------------------------------------------------------

for P in periods_TP_demo:

    data_TP_demo[P] = {}

    for T in temperatures_TP_demo:

        h, m = simulator(
            L=L_TP_demo,
            T=T,
            h0=h0_TP_demo,
            period=P,
            n=n_cycles_TP_demo,
            seed=seed_TP_demo
        )

        h = np.asarray(h)
        m = np.asarray(m)

        # Split magnetization into separate cycles
        m_cycles = m.reshape(n_cycles_TP_demo, P)

        # Remove initial transient cycles
        m_steady = m_cycles[burn_in_TP_demo:]

        # Average response at each driving phase
        m_mean = np.mean(m_steady, axis=0)

        # One complete field cycle
        h_cycle = h[-P:].copy()

        # Mean signed driving work per spin
        dh = np.roll(h_cycle, -1) - h_cycle
        W = -np.sum(m_mean * dh)

        data_TP_demo[P][T] = {
            "h": h_cycle,
            "m_mean": m_mean,
            "m_raw": m_cycles[-1].copy(),
            "h3": h[-3*P:].copy(),
            "m3": m_cycles[-3:].reshape(-1).copy(),
            "W": W
        }
        

    print(
        f"P = {P:3d} completed | "
        f"Elapsed: {time.perf_counter() - start_time:.1f}s",
        flush=True
    )

print("\nAll T-P combinations completed.")

In [ ]:
# ==========================================================
# Multi-Temperature Period Explorer — Part 2
# Three Cycles + Temperature Checkboxes + Period Playback
# ==========================================================

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets

from IPython.display import display
from io import BytesIO
from collections import OrderedDict


# ==========================================================
# 1. Settings
# ==========================================================

periods = [int(P) for P in periods_TP_demo]
temperatures = list(temperatures_TP_demo)

# Consistent colors across both plots
colors = {
    T: plt.cm.tab10(i)
    for i, T in enumerate(temperatures)
}

# Cache rendered images, not Monte Carlo trajectories
image_cache = OrderedDict()
CACHE_LIMIT = 120


# ==========================================================
# 2. Figure Rendering Function
# ==========================================================

def render_TP_figure(P, selected_T):

    selected_T = tuple(selected_T)
    cache_key = (P, selected_T)

    # Reuse an existing image whenever possible
    if cache_key in image_cache:
        image_cache.move_to_end(cache_key)
        return image_cache[cache_key]

    fig, (ax_time, ax_loop) = plt.subplots(
        1, 2,
        figsize=(13, 5)
    )

    # ------------------------------------------------------
    # LEFT: External Field over Three Cycles
    # ------------------------------------------------------

    h3 = np.asarray(
        data_TP_demo[P][temperatures[0]]["h3"]
    )

    t3 = np.arange(3 * P) / P

    ax_time.plot(
        t3,
        h3,
        "k--",
        linewidth=1.6,
        alpha=0.65,
        label="External Field h(t)"
    )

    # ------------------------------------------------------
    # Plot Only Selected Temperatures
    # ------------------------------------------------------

    for T in selected_T:

        data = data_TP_demo[P][T]
        color = colors[T]

        # LEFT: Actual three-cycle response
        m3 = np.asarray(data["m3"])

        ax_time.plot(
            t3,
            m3,
            color=color,
            linewidth=1.7,
            label=f"T = {T:g}"
        )

        # RIGHT: Phase-averaged Hysteresis Loop
        h_cycle = np.asarray(data["h"])
        m_mean = np.asarray(data["m_mean"])

        # Close average loop for visualization
        h_closed = np.r_[h_cycle, h_cycle[0]]
        m_closed = np.r_[m_mean, m_mean[0]]

        ax_loop.plot(
            h_closed,
            m_closed,
            color=color,
            linewidth=2.2,
            label=f"T = {T:g} | W = {data['W']:.2f}"
        )

    # ======================================================
    # 3. LEFT Plot Formatting
    # ======================================================

    ax_time.set_xlim(0, 3)
    ax_time.set_ylim(-1.05, 1.05)

    ax_time.set_xticks([0, 1, 2, 3])

    ax_time.set_xlabel("Time (Driving Cycles)")
    ax_time.set_ylabel("Field h(t) / Magnetization m(t)")

    ax_time.set_title(
        "Magnetization Response — 3 Cycles"
    )

    for boundary in [1, 2]:
        ax_time.axvline(
            boundary,
            color="gray",
            linestyle=":",
            alpha=0.4
        )

    ax_time.axhline(
        0,
        color="gray",
        alpha=0.3
    )

    ax_time.grid(alpha=0.2)
    ax_time.legend(fontsize=9, loc="upper right")

    # ======================================================
    # 4. RIGHT Plot Formatting
    # ======================================================

    ax_loop.set_xlim(
        -1.2 * h0_TP_demo,
        1.2 * h0_TP_demo
    )

    ax_loop.set_ylim(-1.05, 1.05)

    ax_loop.set_xlabel("External Field h")
    ax_loop.set_ylabel("Magnetization m")

    ax_loop.set_title(
        "Hysteresis Loops at Different T"
    )

    ax_loop.axhline(0, color="gray", alpha=0.3)
    ax_loop.axvline(0, color="gray", alpha=0.3)

    ax_loop.grid(alpha=0.2)

    if selected_T:
        ax_loop.legend(
            fontsize=9,
            loc="lower right"
        )
    else:
        ax_loop.text(
            0.5, 0.5,
            "Select a temperature",
            transform=ax_loop.transAxes,
            ha="center",
            va="center",
            fontsize=12,
            color="gray"
        )

    # ======================================================
    # 5. Figure Title
    # ======================================================

    fig.suptitle(
        f"Multi-Temperature Response | "
        f"P = {P} MCS | L = {L_TP_demo}",
        fontsize=14
    )

    fig.tight_layout()

    # ======================================================
    # 6. Save to Memory
    # ======================================================

    buffer = BytesIO()

    fig.savefig(
        buffer,
        format="png",
        dpi=115
    )

    image_bytes = buffer.getvalue()

    buffer.close()
    plt.close(fig)

    # Cache image
    image_cache[cache_key] = image_bytes

    # Limit memory consumption
    if len(image_cache) > CACHE_LIMIT:
        image_cache.popitem(last=False)

    return image_bytes


# ==========================================================
# 7. Temperature Checkboxes
# ==========================================================

temperature_checkboxes = {}

for T in temperatures:

    cb = widgets.Checkbox(
        value=True,
        description=f"T = {T:g}",
        indent=False,
        layout=widgets.Layout(width="105px")
    )

    temperature_checkboxes[T] = cb


def get_selected_temperatures():

    return tuple(
        T for T in temperatures
        if temperature_checkboxes[T].value
    )


# ==========================================================
# 8. Period Slider
# ==========================================================

initial_P = 125 if 125 in periods else periods[0]

period_slider_TP = widgets.SelectionSlider(
    options=[
        (f"P = {P}", P)
        for P in periods
    ],
    value=initial_P,
    description="Period:",
    continuous_update=False,
    layout=widgets.Layout(width="650px")
)


# ==========================================================
# 9. Image Display
# ==========================================================

period_image_TP = widgets.Image(
    value=render_TP_figure(
        initial_P,
        get_selected_temperatures()
    ),
    format="png",
    layout=widgets.Layout(width="100%")
)


# ==========================================================
# 10. Shared Update Function
# ==========================================================

def update_TP_display(change=None):

    P = period_slider_TP.value
    selected_T = get_selected_temperatures()

    period_image_TP.value = render_TP_figure(
        P,
        selected_T
    )


# Slider callback
period_slider_TP.observe(
    update_TP_display,
    names="value"
)

# Checkbox callbacks
for cb in temperature_checkboxes.values():

    cb.observe(
        update_TP_display,
        names="value"
    )


# ==========================================================
# 11. Auto-Play
# ==========================================================

play_TP = widgets.Play(
    value=periods.index(initial_P),
    min=0,
    max=len(periods) - 1,
    step=1,
    interval=450
)

widgets.jslink(
    (play_TP, "value"),
    (period_slider_TP, "index")
)


# ==========================================================
# 12. Optional: Pre-render Selected Configuration
# ==========================================================

preload_button = widgets.Button(
    description="Preload Selected T",
    icon="refresh",
    button_style="",
    tooltip="Cache all periods for the selected temperatures"
)

def preload_selected_T(_):

    selected_T = get_selected_temperatures()

    preload_button.disabled = True

    try:
        for P in periods:
            render_TP_figure(P, selected_T)
    finally:
        preload_button.disabled = False

    update_TP_display()


preload_button.on_click(preload_selected_T)


# ==========================================================
# 13. Display Explorer
# ==========================================================

checkbox_row = widgets.HBox(
    list(temperature_checkboxes.values())
)

control_row = widgets.HBox([
    play_TP,
    period_slider_TP
])

display(
    widgets.VBox([
        widgets.HTML("<b>Temperature Selection</b>"),
        checkbox_row,
        control_row,
        preload_button,
        period_image_TP
    ])
)

In [ ]:
# ==========================================================
# STAGE 3 — Finite-Size Analysis
# A(P) and Q(P) for Different Lattice Sizes
# ==========================================================

import numpy as np
import matplotlib.pyplot as plt
import time

# ==========================================
# 1. Experimental Parameters
# ==========================================

lattice_sizes_finite = [20, 40, 80]

periods_finite = [
    10, 25, 50, 75, 100,
    125, 150, 200, 300
]

seeds_finite = [101, 202, 303]

T_finite = 2.0
h0_finite = 0.5

n_cycles_finite = 25
burn_in_finite = 10


# ==========================================
# 2. Define Observables
# ==========================================

def signed_work_finite(h, m):
    """
    Signed work per spin over one driving cycle.
    m[t] is measured after the sweep at field h[t].
    """
    h = np.asarray(h)
    m = np.asarray(m)

    dh = np.roll(h, -1) - h

    return -np.sum(m * dh)


# Shape: (L, P, seed)
shape_finite = (
    len(lattice_sizes_finite),
    len(periods_finite),
    len(seeds_finite)
)

all_W_finite = np.zeros(shape_finite)
all_Aabs_finite = np.zeros(shape_finite)
all_Q_finite = np.zeros(shape_finite)


# ==========================================
# 3. Run Monte Carlo Simulations
# ==========================================

start_time = time.perf_counter()

for i, L in enumerate(lattice_sizes_finite):

    print(f"\nRunning L = {L}", flush=True)

    for j, P in enumerate(periods_finite):

        for s, seed in enumerate(seeds_finite):

            h, m = simulator(
                L=L,
                T=T_finite,
                h0=h0_finite,
                period=P,
                n=n_cycles_finite,
                seed=seed
            )

            h = np.asarray(h)
            m = np.asarray(m)

            # Divide data into individual cycles
            h_cycles = h.reshape(n_cycles_finite, P)
            m_cycles = m.reshape(n_cycles_finite, P)

            # Discard transient cycles
            h_steady = h_cycles[burn_in_finite:]
            m_steady = m_cycles[burn_in_finite:]

            # ------ Hysteresis Work ------

            cycle_W = []

            for k in range(len(h_steady)):

                W = signed_work_finite(
                    h_steady[k],
                    m_steady[k]
                )

                cycle_W.append(W)

            cycle_W = np.asarray(cycle_W)

            # Signed mean work
            all_W_finite[i, j, s] = np.mean(cycle_W)

            # Old definition: mean absolute area
            all_Aabs_finite[i, j, s] = np.mean(
                np.abs(cycle_W)
            )

            # ------ Dynamic Order Parameter ------

            Q_cycles = np.mean(m_steady, axis=1)

            all_Q_finite[i, j, s] = np.mean(
                np.abs(Q_cycles)
            )

        print(f"  P = {P:3d} completed", flush=True)


print(
    f"\nCompleted in "
    f"{time.perf_counter() - start_time:.2f} seconds"
)


# ==========================================
# 4. Ensemble Statistics
# ==========================================

def ensemble_statistics_finite(data):

    mean = np.mean(data, axis=2)

    sem = (
        np.std(data, axis=2, ddof=1)
        / np.sqrt(data.shape[2])
    )

    return mean, sem


mean_W_finite, error_W_finite = (
    ensemble_statistics_finite(all_W_finite)
)

mean_Aabs_finite, error_Aabs_finite = (
    ensemble_statistics_finite(all_Aabs_finite)
)

mean_Q_finite, error_Q_finite = (
    ensemble_statistics_finite(all_Q_finite)
)


# ==========================================
# 5. Final Plot
# ==========================================

fig, axes = plt.subplots(
    1, 2,
    figsize=(13, 5)
)

# ------ Left: Mean Dissipated Work ------

for i, L in enumerate(lattice_sizes_finite):

    axes[0].errorbar(
        periods_finite,
        mean_W_finite[i],
        yerr=error_W_finite[i],
        fmt='o-',
        capsize=3,
        label=f'L = {L}'
    )

axes[0].set_xlabel("Driving Period P (MCS)")
axes[0].set_ylabel("Mean Signed Work per Spin")
axes[0].set_title("Finite-Size Dependence of Hysteresis")
axes[0].legend()
axes[0].grid(alpha=0.3)


# ------ Right: Dynamic Order Parameter ------

for i, L in enumerate(lattice_sizes_finite):

    axes[1].errorbar(
        periods_finite,
        mean_Q_finite[i],
        yerr=error_Q_finite[i],
        fmt='o-',
        capsize=3,
        label=f'L = {L}'
    )

axes[1].set_xlabel("Driving Period P (MCS)")
axes[1].set_ylabel("Mean |Q|")
axes[1].set_title("Finite-Size Dependence of Dynamic Order")
axes[1].legend()
axes[1].grid(alpha=0.3)

fig.suptitle(
    f"Finite-Size Analysis | T = {T_finite}, "
    f"h0 = {h0_finite}",
    fontsize=14
)

plt.tight_layout()
plt.show()